
<div style="text-align: center; padding: 30px 10px;">

<h1 style="color:#ff7500; font-size: 24px; margin-bottom: 10px;">
МФТИ ФПМИ
</h1>

<h2 style="font-size: 30px; margin-top: 5px;">
Практикум Python - Продвинутый Поток
</h2>

<hr style="width: 60%; border: 1px solid #10069f; margin: 25px auto;">

<h3 style="font-size: 36px;">
12. Asyncio. Threading. Практика.
</h3>

<p style="margin-top: 20px;">
<strong>Дата:</strong> 21 апреля - 23 апреля 2026 года<br>
</p>

<p style="margin-top: 25px;">
Данный ноутбук является частью серии семинаров по курсу  
<em>«Практикум Python»</em> и предназначен для учебных и образовательных целей.
</p>

</div>

## Sync vs Async

`requests` `aiohttp` `web` `asyncio` `threads`

## Условие

Вам предстоит сравнить 2 подхода к распараллеливанию сетевых запросов в питоне: синхронный и асинхронный.

Вам нужно реализовать 2 функции (если точнее, то 1 из 2 - корутина), которые получают урл (адрес страницы)
на вход, делают сетевой запрос и возвращают тело страницы (для простоты игнорируем статусы и считаем, что любое тело,
даже в случае http кодов ошибок 4xx/5xx - это ок). Это делается в одну-две строчки, так что не мудрите.

Дальше вам нужно написать 2 обертки над вашими функциями для конкурентного (concurrent) получения списка страниц:
одна должна создавать N корутин и асинхронно дожидаться их завершения, а вторая должна делать то же самое, используя
пул тредов, чтобы так же распараллелить ожидание загрузки. В случае использования пула тредов обратите внимание, что
кол-во тредов по-умолчанию может быть меньше, чем кол-во урлов в тестах - это может влиять на скорость скачивания.


In [ ]:
import aiohttp
import requests


async def async_fetch(session: aiohttp.ClientSession, url: str) -> str:
    """
    Asyncronously fetch (get-request) single url using provided session
    :param session: aiohttp session object
    :param url: target http url
    :return: fetched text
    """
    async with session.get(url) as s:
        return await s.text()


def sync_fetch(session: requests.Session, url: str) -> str:
    """
    Syncronously fetch (get-request) single url using provided session
    :param session: requests session object
    :param url: target http url
    :return: fetched text
    """
    return session.get(url).text

/Users/admin/Experiments/Python/mipt_python/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [10]:
from concurrent.futures import ThreadPoolExecutor

def sync_requests(urls: list[str]) -> list[str]:
    """
    Fetch provided urls with requests WITHOUT concurrency
    :param urls: list of http urls ot fetch
    :return: list of fetched texts
    """
    session = requests.Session()
    return [sync_fetch(session, url) for url in urls]


async def async_requests(urls: list[str]) -> list[str]:
    """
    Concurrently fetch provided urls using aiohttp
    :param urls: list of http urls ot fetch
    :return: list of fetched texts
    """
    async with aiohttp.ClientSession() as session:
        return [await async_fetch(session, url) for url in urls]

def threaded_requests(urls: list[str]) -> list[str]:
    """
    Concurrently fetch provided urls with requests in different threads
    :param urls: list of http urls ot fetch
    :return: list of fetched texts
    """
    session = requests.Session()
    surls = [(session, url) for url in urls]

    with ThreadPoolExecutor(max_workers=len(urls)) as executor:
        results = executor.map(sync_fetch, surls)
        return list(results)


In [12]:
import asyncio
import time
from functools import partial

import aiohttp
import requests
from aiohttp import web
from aiohttp.test_utils import AioHTTPTestCase
import unittest

# from .sync_vs_async import async_fetch, sync_fetch, async_requests, threaded_requests


class FetchingTestCase(AioHTTPTestCase):
    async def get_application(self) -> web.Application:
        async def hello_handler(request: web.Request) -> web.Response:
            return web.Response(text='Hello, asyncio!')

        app = web.Application()
        app.router.add_get('/', hello_handler)
        return app

    @property
    def server_address(self) -> str:
        return f'http://{self.server.host}:{self.server.port}/'

    async def test_async_fetching(self) -> None:
        async with aiohttp.ClientSession() as session:
            response = await async_fetch(session, self.server_address)
        assert response == 'Hello, asyncio!'

    async def test_sync_fetching(self) -> None:
        with requests.Session() as session:
            response = await asyncio.to_thread(  # a convenient way to wait for a syncronous code
                partial(sync_fetch, session, self.server_address)
            )
        assert response == 'Hello, asyncio!'


class TimedTestCase(AioHTTPTestCase):
    async def get_application(self) -> web.Application:
        async def sleepy_handler(request: web.Request) -> web.Response:
            await asyncio.sleep(1)
            return web.Response(text='OK')

        app = web.Application()
        app.router.add_get('/', sleepy_handler)
        return app

    @property
    def server_address(self) -> str:
        return f'http://{self.server.host}:{self.server.port}/'

    async def test_async_parallel(self) -> None:
        start = time.time()
        responses = await async_requests(
            [self.server_address] * 10
        )
        end = time.time()
        assert end - start <= 1.5
        assert responses == ['OK'] * 10

    async def test_threaded_parallel(self) -> None:
        start = time.time()
        responses = await asyncio.to_thread(
            partial(threaded_requests, [self.server_address] * 10)
        )
        end = time.time()
        assert end - start <= 1.5
        assert responses == ['OK'] * 10




if __name__ == '__main__':
    # 'argv' and 'exit' are required to run within a notebook
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

RuntimeError: Cannot run the event loop while another loop is running

## Async proxy

`aiohttp` `web` `asyncio` `http proxy` `URL`

## Условие

Вам нужно реализовать простенький веб-сервер на aiohttp, который будет работать как http-proxy.

У него должна быть одна ручка: /fetch?url=..., в которую передается параметром http(s)-url, который надо
асинхронно подгрузить и вернуть страницу пользователю (включая http код ответа).

В тестах так же проверяется, что ваш сервер корректно обрабатывает невалидный url.

Кроме того, вам нужно реализовать 2 корутины, которые должны инициализировать ваше приложение, чтобы оно
отвечало на ручку /fetch, а также создать и корректно завершить клиентскую сессию для aiohttp, с помощью
которой ваше приложение и будет асинхронно запрашивать страницу по переданному урлу.

In [ ]:
async def proxy_handler(request: web.Request) -> web.Response:
    """
    Check request contains http url in query args:
        /fetch?url=http%3A%2F%2Fexample.com%2F
    and trying to fetch it and return body with http status.
    If url passed without scheme or is invalid raise 400 Bad request.
    On failure raise 502 Bad gateway.
    :param request: aiohttp.web.Request to handle
    :return: aiohttp.web.Response
    """


async def setup_application(app: web.Application) -> None:
    """
    Setup application routes and aiohttp session for fetching
    :param app: app to apply settings with
    """


async def teardown_application(app: web.Application) -> None:
    """
    Application with aiohttp session for tearing down
    :param app: app for tearing down
    """


In [ ]:
import asyncio
import time

import aiohttp
from aiohttp import web
from aiohttp.test_utils import AioHTTPTestCase, make_mocked_request
from yarl import URL

from .async_proxy import proxy_handler, setup_application, teardown_application


class FetchingTestCase(AioHTTPTestCase):
    async def get_application(self) -> web.Application:
        async def ok_handler(request: web.Request) -> web.Response:
            # https://docs.aiohttp.org/en/latest/web_exceptions.html
            raise web.HTTPOk(text='OK')

        async def fail_handler(request: web.Request) -> web.Response:
            raise web.HTTPInternalServerError(text='Internal server error')

        async def sleep_handler(request: web.Request) -> web.Response:
            await asyncio.sleep(1)
            raise web.HTTPOk(text='OK')

        test_app = web.Application()
        test_app.router.add_get('/ok', ok_handler)
        test_app.router.add_get('/fail', fail_handler)
        test_app.router.add_get('/sleep', sleep_handler)
        return test_app

    def server_address(self, path: str) -> str:
        return str(URL.build(
            scheme='http',
            host=self.server.host,
            port=self.server.port,
            path=path
        ))

    @staticmethod
    def fetch_url(url_to_fetch: str) -> str:
        return str(URL.build(
            path='/fetch',
            query={'url': url_to_fetch}
        ))

    async def test_ok(self) -> None:
        async with aiohttp.ClientSession() as session:
            request = make_mocked_request(
                'GET', self.fetch_url(self.server_address('/ok'))
            )
            request.app['session'] = session  # One can use session inside handler
            try:
                response = await proxy_handler(request)
            except web.HTTPException as exc:
                response = exc
        assert response.body == b'OK'
        assert response.status == 200

    async def test_fail(self) -> None:
        async with aiohttp.ClientSession() as session:
            request = make_mocked_request(
                'GET', self.fetch_url(self.server_address('/fail'))
            )
            request.app['session'] = session
            try:
                response = await proxy_handler(request)
            except web.HTTPException as exc:
                response = exc
        assert response.body == b'Internal server error'
        assert response.status == 500

    async def test_bad_url(self) -> None:
        async with aiohttp.ClientSession() as session:
            request = make_mocked_request('GET', self.fetch_url('bad_url'))
            request.app['session'] = session
            try:
                response = await proxy_handler(request)
            except web.HTTPException as exc:
                response = exc
        assert response.status == 400
        assert response.body == b'Empty url scheme'

    async def test_no_url(self) -> None:
        async with aiohttp.ClientSession() as session:
            request = make_mocked_request('GET', '/fetch')
            request.app['session'] = session
            try:
                response = await proxy_handler(request)
            except web.HTTPException as exc:
                response = exc
        assert response.status == 400
        assert response.body == b'No url to fetch'

    async def test_bad_url_scheme(self) -> None:
        async with aiohttp.ClientSession() as session:
            request = make_mocked_request(
                'GET', self.fetch_url('ftp://example.com/')
            )
            request.app['session'] = session
            try:
                response = await proxy_handler(request)
            except web.HTTPException as exc:
                response = exc
        assert response.status == 400
        assert response.body == b'Bad url scheme: ftp'

    async def test_concurrent(self) -> None:
        async with aiohttp.ClientSession() as session:
            tasks = []
            start = time.time()
            for _ in range(3):
                request = make_mocked_request(
                    'GET', self.fetch_url(self.server_address('/sleep'))
                )
                request.app['session'] = session
                tasks.append(proxy_handler(request))
            responses = await asyncio.gather(*tasks, return_exceptions=True)
            end = time.time()
        assert end - start < 1.5
        assert all(response.status == 200 for response in responses)


class ApplicationTestCase(AioHTTPTestCase):
    async def get_application(self) -> web.Application:
        app = web.Application()
        await setup_application(app)

        async def test_handler(request: web.Request) -> web.Response:
            raise web.HTTPOk(text='pong')

        app.router.add_get('/ping', test_handler)  # For test purposes
        return app

    async def tearDownAsync(self) -> None:
        await teardown_application(self.server.app)

    def server_address(self, path: str, query: dict[str, str] | None = None) -> str:
        return str(URL.build(
            scheme='http',
            host=self.server.host,
            port=self.server.port,
            path=path,
            query=query
        ))

    async def fetch_url(
        self, session: aiohttp.ClientSession, url_to_fetch: str
    ) -> aiohttp.ClientResponse:
        url = str(URL(
            self.server_address(
                path='/fetch',
                query={'url': url_to_fetch}
            )
        ))

        return await session.get(url)

    async def test_application(self) -> None:
        async with aiohttp.ClientSession() as session:
            response = await self.fetch_url(
                session, self.server_address(path='/ping')  # yep, server should fetching itself :)
            )
            assert response.status == 200
            assert await response.text() == 'pong'
            # Ensure application didn't close internal aiohttp session after first request
            response = await self.fetch_url(
                session, self.server_address(path='/ping')
            )
            assert response.status == 200

if __name__ == '__main__':
    # 'argv' and 'exit' are required to run within a notebook
    unittest.main(argv=['first-arg-is-ignored'], exit=False)